# Clase 003 — Git y GitHub para data scientists

**Parte 0 — Prerrequisitos** · Pro Git caps. 2-3.

> 🎯 Usar git como sistema serio de versionado: commits atómicos, branches, PRs, conflictos sin pánico, `.gitignore` para DS.

> ⏱️ ~120 min

## ⚙️ Setup

La mayoría de los ejercicios se hacen en terminal. Este notebook documenta los comandos y verifica el estado del repo desde Python.

In [ ]:
import subprocess
from pathlib import Path

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout.strip() or r.stderr.strip()

print('git version:', run('git --version'))
print('cwd        :', Path.cwd())

## 1️⃣ Modelo mental de git

```
[working tree]  ──git add──▶  [staging area]  ──git commit──▶  [local repo]  ──git push──▶  [remote]
     editas                    preparas                       guardas                    publicas
```

Cada commit es **un nodo en un DAG** identificado por su SHA-1. Las ramas son **punteros móviles** a commits. `HEAD` es el puntero al commit actual.

In [ ]:
# Inspecciona el estado del repo donde corre este notebook
print('--- branch actual ---')
print(run('git branch --show-current'))
print('--- últimos 5 commits ---')
print(run('git log --oneline -5'))
print('--- archivos modificados ---')
print(run('git status -s') or '(working tree limpio)')

## 2️⃣ Commits atómicos + mensajes convencionales

**Atómico** = un commit = un cambio lógico que puede revertirse solo.

**Mensaje convencional**: `tipo(scope): descripción corta`

Tipos comunes:
- `feat` — nueva funcionalidad
- `fix` — bugfix
- `docs` — solo documentación
- `refactor` — refactor sin cambio de comportamiento
- `test` — añade/modifica tests
- `chore` — mantenimiento (deps, build)

Ejemplos buenos:
- `feat(api): agregar endpoint /predict para modelo v2`
- `fix(loader): manejar nulos en columna fecha`
- `docs(readme): aclarar requisitos de instalación`

Ejemplos malos:
- `update` ← ¿qué?
- `fix bug` ← ¿cuál bug?
- `wip` ← no llega a main

## 3️⃣ Branches: crear, mergear, conflicto

```bash
git switch -c feature/nuevo-modelo    # crea y cambia a nueva rama
# … editas, commits …
git switch main
git merge feature/nuevo-modelo        # merge
```

**Conflicto** = git no puede decidir qué versión gana en una línea. Git marca el archivo así:

```
<<<<<<< HEAD
versión de main
=======
versión de la rama
>>>>>>> feature/nuevo-modelo
```

**Resolución**: edita el archivo dejando solo lo que quieres, borra los marcadores, `git add <archivo>`, `git commit` (mensaje pre-rellenado).

**Merge vs rebase** (regla simple): merge para ramas compartidas, rebase solo para tu rama local antes de PR.

## 4️⃣ `.gitignore` para data science

Las reglas que **siempre** van en un proyecto de DS:

```gitignore
# Entornos
.venv/
venv/
__pycache__/
*.pyc

# Notebooks
.ipynb_checkpoints/
# (opcional) limpiar outputs: usa nbstripout en pre-commit

# Datos
data/raw/*
data/interim/*
!data/raw/.gitkeep
!data/interim/.gitkeep

# Modelos y artefactos
models/*.pkl
models/*.joblib
*.h5

# Secretos
.env
.env.*
!.env.example

# IDE
.vscode/
.idea/
.DS_Store
```

**Regla de oro:** todo lo que pese >100 MB o sea sensible NUNCA al repo. Para datos versionados usa DVC (clase 159).

In [ ]:
# Demo: simular qué se commitearía
ignored_examples = ['.venv/lib/site-packages/numpy.py', 'data/raw/customers.csv', '.env', 'models/v2.pkl', '.DS_Store']
respected = ['src/loader.py', 'README.md', 'tests/test_loader.py', 'data/raw/.gitkeep']

print('❌ Estos NO deben aparecer en git status:')
for f in ignored_examples:
    print(f'   {f}')
print()
print('✅ Estos SÍ deben aparecer:')
for f in respected:
    print(f'   {f}')

## 5️⃣ Pull Requests con `gh`

```bash
# Una vez por máquina
gh auth login

# En tu repo, después de push
gh pr create --title "feat: nuevo modelo de churn" --body "## Resumen\n- ..."
gh pr list
gh pr view 12 --web
gh pr merge 12 --squash
```

El PR es donde ocurre la **revisión técnica**. Un PR bueno:
- Hace UNA cosa.
- Tiene descripción del *por qué* (el qué ya está en el diff).
- Pasa CI antes de pedir review.
- Incluye screenshots / outputs si es visual.

## 6️⃣ `git reflog` — la red de seguridad

**Borraste una rama por error. Tranquilo.** Git guarda referencias al HEAD durante ~90 días:

```bash
git reflog                           # lista todo lo que ha sido HEAD
# Encuentra el SHA de tu commit perdido
git switch -c rescate <sha>           # nueva rama desde ese punto
```

Mientras no hayas hecho `git gc` agresivo, casi nada se pierde.

In [ ]:
# Demo: muestra tus últimas 5 entradas del reflog
print(run('git reflog -5'))

## ✅ Checklist

- [ ] Mis commits son atómicos y tienen mensajes convencionales
- [ ] Sé crear branches, mergear y resolver un conflicto
- [ ] Mi `.gitignore` cubre `.venv/`, datos, secrets y outputs
- [ ] Sé abrir y mergear un PR con `gh`
- [ ] Sé que `git reflog` existe y para qué sirve

## 📝 Homework

Ver `README.md`. Repo público en GitHub con 5+ commits convencionales, branch mergeada, `.gitignore` de DS y 1 PR cerrado.

## 🔗 Referencias

- [Pro Git book](https://git-scm.com/book) — gratis online
- [Conventional Commits](https://www.conventionalcommits.org/)

➡️ **Siguiente:** [004 — Estructura reproducible de proyecto](../004-estructura-reproducible-de-proyecto-cookiecutter-data-science/README.md)